# Notebook 00 — DERMA-ACT V2 Project Overview and Quick Start

**Purpose:** Introduce the DERMA-ACT V2 research framework, verify the local Python environment and installed packages, inspect the repository layout, validate Hydra YAML configurations, run canonical API smoke checks, and explain ISIC 2019 preparation.  
**Data & Artifact Requirement:** **None (Clean-Room Standalone).** Runs top-to-bottom without raw ISIC 2019 imagery, model checkpoints (`.ckpt`, `.pt`), or per-image `.npz` caches.

> **Research Software Only:** DERMA-ACT V2 accompanies a retrospective computational comparative study on ISIC 2019. It is not clinically validated and is not intended for clinical diagnosis or patient care.

## 1. Learning Objectives & Scientific Context

### Learning Objectives
1. Locate the DERMA-ACT repository root portably without hardcoded workstation paths.
2. Verify Python, PyTorch, CUDA, and scientific/notebook package versions.
3. Inspect the reorganized repository layout, Hydra configuration presets, and runnable CLI scripts.
4. Execute lightweight architectural and mathematical parity smoke checks directly against `research.derma_act.core` and `src.modules.active_perception`.

### Scientific Context
DERMA-ACT V2 evaluates seven-class dermoscopic classification (`MEL`, `NV`, `BCC`, `AKIEC`, `BKL`, `DF`, `VASC`) on retrospective ISIC 2019 partitions (`N = 25,331`; `13,931` operational grouping units), comparing a single global view (`A0`) against sequential deterministic virtual-view policies (`A1`–`A9`), fixed-sequence comparators, post-hoc temperature scaling, and split-conformal / image-level Mondrian prediction sets.

### Prerequisites
- Python `>= 3.11` (verified on `3.11.9`)
- Core packages from `requirements.txt` and notebook packages from `requirements-notebooks.txt`

## 2. Environment & Dependency Verification

In [ ]:
from __future__ import annotations

import json
import os
import platform
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


def find_repo_root(start: Path | None = None) -> Path:
    """Locate repository root portably without hardcoded machine paths."""
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "research" / "derma_act" / "config.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate DERMA-ACT repository root containing pyproject.toml and research/derma_act/config.py")


def find_tables_dir(repo_root: Path) -> Path | None:
    for rel in [
        Path("supplementary") / "tables",
        Path("manuscripts") / "current" / "PAPER_DRAFT_Q2_V3_2_1_PRE_SUBMISSION" / "tables",
    ]:
        cand = repo_root / rel
        if cand.is_dir():
            return cand
    return None


SEED = 42
random.seed(SEED)
np.random.seed(SEED)

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import importlib.metadata
import torch

OUT_DIR = REPO_ROOT / "outputs" / "notebooks" / "nb00_quickstart"
OUT_DIR.mkdir(parents=True, exist_ok=True)

packages = [
    "torch", "torchvision", "numpy", "scipy", "scikit-learn",
    "pandas", "matplotlib", "opencv-python", "albumentations",
    "Pillow", "PyYAML", "pydantic", "pytest", "nbformat",
]
env_rows = []
for pkg in packages:
    try:
        ver = importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        ver = "NOT_INSTALLED"
    env_rows.append({"package": pkg, "version": ver})

print(f"Repository Root : {REPO_ROOT.name} (resolved cleanly)")
print(f"Python Version  : {platform.python_version()} ({platform.system()} {platform.machine()})")
print(f"CUDA Available  : {torch.cuda.is_available()} (PyTorch CUDA={torch.version.cuda})")
print(f"Output Dir      : {OUT_DIR.relative_to(REPO_ROOT)}")
pd.DataFrame(env_rows)

## 3. Repository Structure & Artifact Availability Check

In [ ]:
core_dirs = ["src", "research", "scripts", "configs", "tests", "notebooks", "data", "provenance", "docs", "artifacts"]
dir_rows = []
for d in core_dirs:
    p = REPO_ROOT / d
    n_files = sum(1 for f in p.rglob("*") if f.is_file() and "__pycache__" not in f.parts) if p.is_dir() else 0
    dir_rows.append({"directory": d, "exists": p.is_dir(), "has_readme": (p / "README.md").is_file(), "file_count": n_files})

artifact_checks = [
    ("Split metadata (tracked)", Path("data/isic2019/splits/split_metadata.json"), "TRACKED"),
    ("Risk-cost aggregate CSV (tracked)", Path("data/aggregate/FIGURE_4_V1_1_DATA.csv"), "TRACKED"),
    ("Frozen lambda JSON (tracked)", Path("research/derma_act/runs/derma_act_v1/policy/lambda_selection.json"), "TRACKED"),
    ("Frozen tau JSON (tracked)", Path("research/derma_act/runs/derma_act_v1/policy/stopping_threshold.json"), "TRACKED"),
    ("ISIC 2019 raw metadata CSV (local)", Path("data/isic2019/raw/ISIC_2019_Training_Metadata.csv"), "REQUIRES_DATA"),
    ("Frozen A0 test logits NPZ (local)", Path("research/baseline_v2/runs/clean_run_001/predictions/test_canonical.npz"), "REQUIRES_MODEL_ARTIFACT"),
    ("Frozen policy weights policy.pt (local)", Path("research/derma_act/runs/derma_act_v1/policy/policy.pt"), "REQUIRES_MODEL_ARTIFACT"),
]
art_df = pd.DataFrame([
    {"artifact": name, "relative_path": rel.as_posix(), "tier": tier, "present_locally": (REPO_ROOT / rel).is_file()}
    for name, rel, tier in artifact_checks
])

print("=== Active Directory Inventory ===")
display(pd.DataFrame(dir_rows))
print("=== Key Artifact Availability ===")
display(art_df)

## 4. Methodology & Configuration Overview

DERMA-ACT V2 separates reusable neural/operator primitives (`src/`) from pinned scientific constants and canonical mathematical definitions (`research/derma_act/`):
- **State representation ($s_t \in \mathbb{R}^{28}$):** 7 fused class probabilities + Shannon entropy ($\varepsilon = 10^{-12}$) + confidence ($\max p_t$) + top-2 margin + 9-D binary history + 9-D remaining mask (`history[k] + mask[k] == 1`).
- **Recursive arithmetic probability fusion:** $p_{t+1} = (b \cdot p_t + p_{\mathrm{cand}}) / (b + 1)$ for $b \ge 1$ fused observations (explicit running arithmetic mean, **not** Bayesian posterior updating).
- **Supervised surrogate policy:** `MultiHeadActionPolicy` (`37 -> 64 -> 64` + 3 scalar heads = `6,787` parameters) estimates one-step utility $U = \lambda_Q \Delta \hat{Q} + \lambda_C \Delta \hat{C} - \lambda_I \sigma(\mathrm{logit}_i) - \lambda_K \cdot 1.0$, stopping when $\max_a U(s_t, a) < \tau$.

Below we parse all YAML files under `configs/`, inspect available top-level scripts under `scripts/`, and verify canonical API shapes deterministically.

In [ ]:
import yaml

yaml_files = sorted((REPO_ROOT / "configs").rglob("*.yaml"))
cfg_rows = []
for yf in yaml_files:
    parsed = yaml.safe_load(yf.read_text(encoding="utf-8"))
    top_keys = list(parsed.keys()) if isinstance(parsed, dict) else [type(parsed).__name__]
    cfg_rows.append({"config_file": yf.relative_to(REPO_ROOT).as_posix(), "top_level_keys": ", ".join(top_keys[:6])})

scripts_top = sorted(p.relative_to(REPO_ROOT).as_posix() for p in (REPO_ROOT / "scripts").glob("*.py"))
print(f"Validated {len(cfg_rows)} YAML config files; found {len(scripts_top)} portable top-level Python scripts:")
for s in scripts_top:
    print(" -", s)
pd.DataFrame(cfg_rows)

In [ ]:
from research.derma_act.config import (
    ACTION_DISPLAY, BUDGET, CANDIDATE_DIM, CLASS_NAMES, EPS,
    EXPECTED_HASHES, N_CLASSES, STATE_DIM, load_lambda, load_tau,
)
from research.derma_act.core.fusion import fuse_posterior
from research.derma_act.core.state import construct_state
from research.derma_act.core.stopping import should_stop
from src.modules.active_perception.policy import MultiHeadActionPolicy

lam = load_lambda()
tau = load_tau()
policy = MultiHeadActionPolicy(state_dim=STATE_DIM, action_dim=CANDIDATE_DIM, hidden_dim=64)
n_params = sum(p.numel() for p in policy.parameters())

# Architectural shape check using a deterministic uniform test vector (NOT a benchmark metric)
p_uniform = np.full(N_CLASSES, 1.0 / N_CLASSES, dtype=np.float32)
p_fused = fuse_posterior(p_uniform, p_uniform, num_fused=1)
state_vec, ent, conf, margin = construct_state(p_fused, np.zeros(CANDIDATE_DIM), np.ones(CANDIDATE_DIM))

with torch.no_grad():
    dq_hat, dc_hat, flip_logit = policy(
        torch.from_numpy(state_vec).unsqueeze(0),
        torch.eye(CANDIDATE_DIM, dtype=torch.float32)[:1],
    )

assert n_params == 6787, f"Expected 6787 params, got {n_params}"
assert state_vec.shape == (28,)
assert np.allclose(p_fused, p_uniform)
assert should_stop(-0.5, tau) is True and should_stop(0.0, tau) is False

summary = pd.DataFrame([
    {"parameter": "CLASS_NAMES", "value": ", ".join(CLASS_NAMES)},
    {"parameter": "STATE_DIM / CANDIDATE_DIM / BUDGET", "value": f"{STATE_DIM} / {CANDIDATE_DIM} / {BUDGET}"},
    {"parameter": "Frozen lambda (Q, C, I, K)", "value": str(lam)},
    {"parameter": "Frozen stopping tau", "value": f"{tau:.6f} (-3/7)"},
    {"parameter": "MultiHeadActionPolicy parameters", "value": f"{n_params:,}"},
    {"parameter": "Forward output shapes (dq, dc, flip)", "value": f"{tuple(dq_hat.shape)}, {tuple(dc_hat.shape)}, {tuple(flip_logit.shape)}"},
    {"parameter": "Canonical entropy EPS", "value": str(EPS)},
])
display(summary)

# Visualize architecture parameter breakdown across layers (real model weights count, not performance data)
layer_counts = [(name, p.numel()) for name, p in policy.named_parameters()]
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh([x[0] for x in layer_counts], [x[1] for x in layer_counts], color="#2b6cb0")
ax.set_xlabel("Trainable Parameters")
ax.set_title("MultiHeadActionPolicy Parameter Distribution (Total = 6,787)")
ax.invert_yaxis()
for i, (_, cnt) in enumerate(layer_counts):
    ax.text(cnt + 30, i, str(cnt), va="center", fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / "policy_param_breakdown.png", dpi=150)
plt.show()

## 5. Dataset Preparation Overview, Interpretation & Limitations

### Preparing ISIC 2019 Locally (`REQUIRES_DATA`)
Raw ISIC 2019 dermoscopic images (`CC-BY-NC 4.0`) are **not** bundled in Git. To run data-dependent scripts and optional raw-data notebook cells:
1. Acquire ISIC 2019 Training Input, GroundTruth CSV, and Metadata CSV from `https://challenge.isic-archive.com/data/#2019`.
2. Place CSVs under `data/isic2019/raw/` and images under `data/isic2019/ISIC_2019_Training_Input/`.
3. Preserve mandatory `CC-BY-NC 4.0` attributions for `BCN_20000`, `HAM10000`, and `MSK`.

### Important Interpretation & Scope Notes
- The forward pass executed above uses an **untrained** `MultiHeadActionPolicy` instance solely to verify tensor shapes and the `6,787`-parameter architecture; it is **not** a diagnostic accuracy result.
- Static YAML defaults in `configs/train.yaml` (`lr = 1e-3`, `loss = focal`) differ from the verified executed training protocol (`lr = 1e-4`, `CrossEntropyLoss(label_smoothing=0.1)`); always consult `configs/README.md` and `supplementary/tables/tab_s1_classifier_protocol.tex`.

### Reproduction Information
- Deterministic seed: `SEED = 42`
- Equivalent CLI verification (`VERIFIED`): `python -m pytest -p no:cacheprovider -v tests/test_canonical_math.py tests/test_release_smoke.py`
- Output saved to: `outputs/notebooks/nb00_quickstart/policy_param_breakdown.png`

### Next Notebooks & References
- Next: [`01_ISIC2019_Data_and_Split_Audit.ipynb`](01_ISIC2019_Data_and_Split_Audit.ipynb) · [`02_Baseline_Classifier_Evaluation.ipynb`](02_Baseline_Classifier_Evaluation.ipynb) · [`03_Sequential_View_Policy_Analysis.ipynb`](03_Sequential_View_Policy_Analysis.ipynb)
- Docs: [Root README](../README.md) · [Architecture](../docs/ARCHITECTURE.md) · [Configs](../configs/README.md) · [Data Governance](../data/README.md)